In [ ]:
# Uninstall the PEFT library if it causes compatibility issues with transformers
# (e.g., if you're not using LoRA or adapter-based fine-tuning)
!pip uninstall -y peft

In [ ]:
# Install specific PyTorch + CUDA 12.1 compatible versions
# (This matches with Colab's A100 GPUs and avoids binary mismatches)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121 -q

# Uninstall any potentially incompatible versions of transformers and accelerate
!pip uninstall -y transformers accelerate

# Install stable, compatible versions of Hugging Face libraries
!pip install transformers==4.41.2 accelerate==0.30.1 datasets evaluate rouge_score -q

# Install utility for handling JSON Lines (used in your datasets)
!pip install jsonlines


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# This cell is for testing the performance of a finetuned model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Set path or Hugging Face model name
# it can be for example "t5-small", "facebook/bart-base" ή local path π.χ. "./my_model_dir"
#model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum/checkpoint-31025"  # άλλαξε αν χρειάζετα
#model_checkpoint = "GanjinZero/biobart-base"
#model_checkpoint = "QizhiPei/biot5-base"
model_checkpoint = "GanjinZero/biobart-v2-base"


# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to("cuda")

# Prompt for summarization(prompt)
input_text = """Breast cancer (BC) is the most frequent cancer diagnosed in women worldwide. This heterogeneous disease can be classified into four molecular subtypes (luminal A, luminal B, HER2 and triple-negative breast cancer (TNBC)) according to the expression of the estrogen receptor (ER) and the progesterone receptor (PR), and the overexpression of the human epidermal growth factor receptor 2 (HER2). Current BC treatments target these receptors (endocrine and anti-HER2 therapies) as a personalized treatment. Along with chemotherapy and radiotherapy, these therapies can have severe adverse effects and patients can develop resistance to these agents. Moreover, TNBC do not have standardized treatments. Hence, a deeper understanding of the development of new treatments that are more specific and effective in treating each BC subgroup is key. New approaches have recently emerged such as immunotherapy, conjugated antibodies, and targeting other metabolic pathways. This review summarizes current BC treatments and explores the new treatment strategies from a personalized therapy perspective and the resulting challenges. """

# Tokenization and creation of input IDs
inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True).to("cuda")

# Summary creation
summary_ids = model.generate(inputs, max_new_tokens=100, num_beams=4, early_stopping=True)

# Decoding and printing
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"\n📝 Summary:\n{summary}")


In [ ]:
# Import core libraries
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)
from datasets import load_dataset, load_from_disk, Dataset
import evaluate

# Utilities
import pandas as pd
import matplotlib.pyplot as plt
import jsonlines
import numpy as np
from tqdm import tqdm
import os

# Environment checks
print(f" Torch version: {torch.__version__}")

import transformers
print(f" Transformers version: {transformers.__version__}")

# Check if CUDA is available and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {device}")


In [ ]:
model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum/checkpoint-31025"

In [ ]:
# =========================
#  Model Selection & Checkpoint Configuration
# =========================

# Choose model type: either 'biot5' or 'biobart'
model_choice = "biot5"  # Change this to "biobart" if needed

# Define the corresponding HuggingFace checkpoint
if model_choice == "biot5":
    model_checkpoint = "QizhiPei/biot5-base"
elif model_choice == "t5":
    model_checkpoint = "t5-base"
elif model_choice == "biov2bart":
    model_checkpoint = "GanjinZero/biobart-v2-base"
elif model_choice == "bart":
    model_checkpoint = "facebook/bart-base"
else:
    raise ValueError(" Invalid model choice! Use 'biot5', 'biobart' or biov2bart.")

print(f" Selected model checkpoint: {model_checkpoint}")



In [ ]:
# =========================
# Load Tokenizer & Model
# =========================

# Load the tokenizer corresponding to the selected model
# Responsible for converting text to input token IDs (and back)
#tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Load the pretrained Seq2Seq model (e.g., T5 or BART)
# This model is designed for generation tasks like summarization
#model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

print(f" Model and tokenizer loaded to device: {device}")

In [ ]:
from datasets import load_from_disk

# =========================
#  Load Tokenized Datasets
# =========================

if model_choice == "biot5":
    train_path = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/biot5_sum/train"
elif model_choice == "t5":
    train_path = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/t5_sum/train"
elif model_choice == "bart":
    train_path = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/bart_sum/train"
elif model_choice == "biov2bart":
    train_path = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/biov2bart_sum/train"


# Load FULL training dataset
full_train_dataset = load_from_disk(train_path)

# =========================
#  Create 90/10 Split
# =========================
split_dataset = full_train_dataset.train_test_split(
    test_size=0.1,   # 10% for validation
    seed=42,
    shuffle=True
)

train_dataset = split_dataset["train"]
validation_dataset = split_dataset["test"]

print(f" Train size: {len(train_dataset)}")
print(f" Validation size: {len(validation_dataset)}")

In [ ]:
# Select only 128 first records for testing
train_dataset = train_dataset.select(range(512))
validation_dataset = validation_dataset.select(range(128))

In [ ]:
if model_choice == "biot5":
    out_dir="/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum"
elif model_choice == "bart":
    out_dir="/content/drive/MyDrive/biomedical_text_generation/models/bart_sum"
elif model_choice == "t5":
    out_dir="/content/drive/MyDrive/biomedical_text_generation/models/t5_sum"
elif model_choice == "biov2bart":
    out_dir="/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum"
    # Where to save model checkpoints and logs


# Define the training arguments
training_args = Seq2SeqTrainingArguments(

    output_dir= out_dir,
    # Where to save model checkpoints and logs

    eval_strategy="epoch",       # Evaluate after each epoch
    save_strategy="epoch",             # Save a checkpoint after each epoch

    learning_rate=1e-5,                # Small learning rate for fine-tuning
    per_device_train_batch_size=16,    # Training batch size per device
    per_device_eval_batch_size=16,     # Evaluation batch size per device
    gradient_accumulation_steps=2,     # Gradient accumulation for effective batch size

    num_train_epochs=100,              # Number of total training epochs
    weight_decay=0.01,                 # Regularization to prevent overfitting

    save_total_limit=3,                # Keep only the 2 most recent checkpoints

    predict_with_generate=True,        # Use generation during validation for summarization
    generation_max_length=256,         # Max length for generated sequences

    logging_dir="/content/drive/MyDrive/biomedical_text_generation/outputs/05_fine_tuning/summarization/biot5_sum",
    logging_strategy="epoch",          # Log once per epoch

    load_best_model_at_end=True,       # Automatically load best checkpoint (lowest eval loss)
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,

    report_to="none",                  # No external logging (e.g., wandb)
    fp16=True                          # Enable mixed-precision training if using CUDA
)

print(" Training arguments configured.")


In [ ]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    labels = np.array(labels)

    # If logits slip through, convert to ids
    if predictions.ndim == 3:
        predictions = predictions.argmax(-1)

    # Replace ignore index in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Remove any negative ids (e.g., -1) before decoding
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
    labels      = np.where(labels < 0, tokenizer.pad_token_id, labels)

    # Clamp to vocab range
    vocab_size = len(tokenizer)
    predictions = np.clip(predictions, 0, vocab_size - 1)
    labels      = np.clip(labels, 0, vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(predictions.tolist(), skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels.tolist(), skip_special_tokens=True)

    # Fix None values
    decoded_preds = [p if isinstance(p, str) else "" for p in decoded_preds]
    decoded_labels = [l if isinstance(l, str) else "" for l in decoded_labels]

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    rouge["bleu"] = bleu["bleu"]
    return {k: round(v, 4) for k, v in rouge.items()}

In [ ]:
# ============================
# Trainer Initialization
# ============================

trainer = Seq2SeqTrainer(
    model=model,  # The Seq2Seq model to be fine-tuned (e.g., T5, BART)
    args=training_args,  # Training hyperparameters and behavior (batch size, epochs, save strategy etc.)

    train_dataset=train_dataset,         # Tokenized training data
    eval_dataset=validation_dataset,     # Tokenized validation data

    tokenizer=tokenizer,                 # Tokenizer used to preprocess data and decode predictions

    compute_metrics=compute_metrics,     # Custom metric function to evaluate the model (e.g., ROUGE)

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5)
        # Early stopping to prevent overfitting.
        # Training stops if validation loss does not improve for 3 consecutive evaluations.
    ]
)

# from transformers import DataCollatorForSeq2Seq

# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=model,   # important for correct label padding (-100)
# )

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=validation_dataset,
#     tokenizer=tokenizer,
#     data_collator=data_collator,   # fixes the error
#     compute_metrics=compute_metrics,
#     callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
# )


In [ ]:
# ===============================
# Start Fine-Tuning
# ===============================
train_result = trainer.train(resume_from_checkpoint=False)  # Starts the training loop using the trainer configuration

# ===============================
# Save Final Model & Tokenizer
# ===============================
# Save both the trained model and tokenizer for later use
if model_choice == "biot5":
    trainer.save_model("/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final")
    tokenizer.save_pretrained("/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final")
elif model_choice == "biobart":
    trainer.save_model("/content/drive/MyDrive/biomedical_text_generation/models/biobart_sum_final")
    tokenizer.save_pretrained("/content/drive/MyDrive/biomedical_text_generation/models/biobart_sum_final")
elif model_choice == "biov2bart":
    trainer.save_model("/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum_final")

print(" Fine-tuning completed and the final model has been saved.")


In [ ]:
# Paths for Saving Outputs
logs_dir = f"/content/drive/MyDrive/biomedical_text_generation/data/plots/summarization/{model_choice}"
images_dir = os.path.join(logs_dir, "images")
os.makedirs(images_dir, exist_ok=True)

CSV_LOG_PATH = os.path.join(logs_dir, "metrics_logs.csv")
PLOT_LOSS_PATH = os.path.join(images_dir, "loss_plot.png")
PLOT_ROUGE_PATH = os.path.join(images_dir, "rouge_plot.png")
PLOT_BLEU_PATH = os.path.join(images_dir, "bleu_plot.png")  # New

# Convert Trainer Logs to DataFrame
log_df = pd.DataFrame(trainer.state.log_history)

# Keep only rows that include 'epoch' (skip early stopping, lr logs etc.)
log_df = log_df[log_df["epoch"].notnull()].copy()

# Group by epoch and average values (to plot 1 point per epoch)
grouped_df = log_df.groupby("epoch").mean(numeric_only=True).reset_index()

# Save full raw log history
log_df.to_csv(CSV_LOG_PATH, index=False)

# Plot: Training & Validation Loss
plt.figure(figsize=(10, 6))
plt.plot(grouped_df["epoch"], grouped_df["loss"], label="Train Loss", marker="o")
plt.plot(grouped_df["epoch"], grouped_df["eval_loss"], label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid()
plt.savefig(PLOT_LOSS_PATH)
plt.show()

# Plot: ROUGE Scores
plt.figure(figsize=(10, 6))
plt.plot(grouped_df["epoch"], grouped_df["eval_rouge1"], label="ROUGE-1", marker="o")
plt.plot(grouped_df["epoch"], grouped_df["eval_rouge2"], label="ROUGE-2", marker="o")
plt.plot(grouped_df["epoch"], grouped_df["eval_rougeL"], label="ROUGE-L", marker="o")
plt.xlabel("Epoch")
plt.ylabel("ROUGE Score")
plt.title("ROUGE Scores over Epochs")
plt.legend()
plt.grid()
plt.savefig(PLOT_ROUGE_PATH)
plt.show()

# Plot: BLEU Score
if "eval_bleu" in grouped_df.columns:
    plt.figure(figsize=(10, 6))
    plt.plot(grouped_df["epoch"], grouped_df["eval_bleu"], label="BLEU", marker="o", color="purple")
    plt.xlabel("Epoch")
    plt.ylabel("BLEU Score")
    plt.title("BLEU Score over Epochs")
    plt.legend()
    plt.grid()
    plt.savefig(PLOT_BLEU_PATH)
    plt.show()
else:
    print(" 'eval_bleu' not found in logs. Skipping BLEU plot.")

# Save file paths
print(f" Metrics CSV saved at: {CSV_LOG_PATH}")
print(f" Loss plot saved at: {PLOT_LOSS_PATH}")
print(f" ROUGE plot saved at: {PLOT_ROUGE_PATH}")
print(f" BLEU plot saved at: {PLOT_BLEU_PATH}")


In [ ]:
import os
import signal

# Forcefully terminate the Colab runtime
os.kill(os.getpid(), signal.SIGKILL)